# 01 - Day of Week Seasonality

Does Bitcoin or Ethereum behave differently on different days?

**What we test:**

- Mean return per weekday (Mon-Sun)
- Volatility per weekday
- Hit rate (fraction of green days)
- **Statistical significance** (t-test: is this real or noise?)

**Verdict rule:** if p < 0.05 the pattern is statistically real. Otherwise it's noise and you should not trade on it.

## Setup

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

pd.set_option('display.width', 180)
plt.rcParams['figure.figsize'] = (12, 4)

DB = Path(r'D:\\Projects\\done\\eth_bot\\eth_bot_v2') / 'database' / 'market.db'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

def load_close(symbol, timeframe='1d'):
    d = sql("SELECT open_time, close FROM ohlcv "
            "WHERE symbol=? AND timeframe=? ORDER BY open_time",
            symbol=symbol, timeframe=timeframe)
    d['date'] = pd.to_datetime(d.open_time, unit='ms', utc=True).tz_convert(None)
    return d.set_index('date')['close']


def ttest(series, name):
    if len(series) < 5:
        return {'name': name, 'mean': series.mean(), 't': np.nan, 'p': np.nan, 'n': len(series), 'sig': False}
    t, p = stats.ttest_1samp(series, 0)
    return {'name': name, 'mean': series.mean(), 't': float(t), 'p': float(p), 'n': len(series), 'sig': bool(p < 0.05)}


## 1. Load data and compute weekday

In [2]:
data = {}
for sym in SYMBOLS:
    c = load_close(sym, '1d')
    df = pd.DataFrame({'close': c})
    df['ret'] = df['close'].pct_change()
    df['weekday'] = df.index.dayofweek   # 0 = Monday, 6 = Sunday
    data[sym] = df.dropna()

for sym, df in data.items():
    print(f'{sym}: {len(df)} days  {df.index.min().date()} -> {df.index.max().date()}')

DatabaseError: Execution failed on sql 'SELECT open_time, close FROM ohlcv WHERE symbol=? AND timeframe=? ORDER BY open_time': Binding 1 has no name, but you supplied a dictionary (which has only names).

## 2. Mean return and volatility by weekday

In [ ]:
WEEKDAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

for sym, df in data.items():
    print(f'\n=== {sym} ===')
    stats_df = df.groupby('weekday')['ret'].agg(
        mean='mean', median='median', std='std', count='count')
    stats_df['hit_rate'] = df.groupby('weekday')['ret'].apply(lambda s: (s > 0).mean())
    stats_df.index = WEEKDAYS
    stats_df['mean_%'] = stats_df['mean'] * 100
    stats_df['std_%'] = stats_df['std'] * 100
    print(stats_df[['mean_%', 'median', 'std_%', 'hit_rate', 'count']].round(4).to_string())

## 3. Bar chart - mean return by weekday

Green bars = historically positive days. Red = negative.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, sym in zip(axes, SYMBOLS):
    df = data[sym]
    means = df.groupby('weekday')['ret'].mean() * 100
    colors = ['seagreen' if v > 0 else 'crimson' for v in means.values]
    ax.bar(WEEKDAYS, means.values, color=colors)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{sym} - mean return by weekday')
    ax.set_ylabel('mean return %')

plt.suptitle('Is Monday different from Friday?', fontsize=13)
plt.tight_layout(); plt.show()

## 4. Volatility by weekday

**What this tells you:** are some days systematically more volatile?
High-vol days = bigger wins AND bigger losses. Low-vol days = safer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, sym in zip(axes, SYMBOLS):
    df = data[sym]
    stds = df.groupby('weekday')['ret'].std() * 100
    colors = plt.cm.viridis(stds / stds.max())
    ax.bar(WEEKDAYS, stds.values, color=colors)
    ax.set_title(f'{sym} - volatility by weekday')
    ax.set_ylabel('std of daily returns %')

plt.suptitle('Are weekends actually quieter than weekdays?', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Hit rate - fraction of positive days

**What this tells you:** even if mean return is positive, if only 40% of days are green, that positive mean comes from a few huge days. Hit rate > 55% is a real directional edge.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, sym in zip(axes, SYMBOLS):
    df = data[sym]
    hit = df.groupby('weekday')['ret'].apply(lambda s: (s > 0).mean())
    ax.bar(WEEKDAYS, hit.values, color='steelblue')
    ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='coin flip (50%)')
    ax.set_ylim(0.4, 0.6)
    ax.set_title(f'{sym} - hit rate by weekday')
    ax.set_ylabel('P(return > 0)')
    ax.legend()

plt.suptitle('Are some days more often green than red?', fontsize=13)
plt.tight_layout(); plt.show()

## 6. Statistical significance test

For each weekday we ask: **is the mean return significantly different from zero?**

- p < 0.05: pattern is real, worth trading
- p > 0.05: pattern is noise, ignore

**Warning:** with 7 weekdays × 2 coins = 14 tests, expect ~1 false positive just by chance. Apply a stricter threshold (p < 0.01) if you want high conviction.

In [ ]:
results = []
for sym, df in data.items():
    for wd in range(7):
        r = df[df.weekday == wd]['ret']
        res = ttest(r, WEEKDAYS[wd])
        res['symbol'] = sym
        results.append(res)

res_df = pd.DataFrame(results)[['symbol', 'name', 'mean', 't', 'p', 'n', 'sig']]
res_df['mean_%'] = res_df['mean'] * 100
res_df['mean_%'] = res_df['mean_%'].round(4)
res_df['t'] = res_df['t'].round(2)
res_df['p'] = res_df['p'].round(4)
print(res_df[['symbol', 'name', 'mean_%', 't', 'p', 'n', 'sig']].to_string(index=False))

## 7. Verdict

The final answer: is any weekday pattern real enough to trade?

In [ ]:
print('=' * 72)
print('VERDICT - DAY OF WEEK SEASONALITY')
print('=' * 72)
print()

for sym in SYMBOLS:
    s = res_df[res_df.symbol == sym]
    sig = s[s.sig]
    print(f'{sym}:')
    if len(sig) > 0:
        print(f'  STATISTICALLY SIGNIFICANT:')
        for _, r in sig.iterrows():
            direction = 'positive' if r['mean'] > 0 else 'negative'
            print(f"    {r['name']}: mean {r['mean_%']:+.4f}%  (p={r['p']:.4f}, {direction})")
    else:
        print(f'  No weekday has statistically significant return')
        best = s.loc[s['mean'].idxmax()]
        print(f"  Best weekday is {best['name']} at {best['mean_%']:+.4f}% (but p={best['p']:.4f}, not significant)")

print()
print('=' * 72)
print('INTERPRETATION')
print('=' * 72)
print()
print('If none are significant (typical for crypto):')
print('  -> Day of week has NO tradeable edge on BTC/ETH')
print('  -> Do not build a strategy around "buy Monday, sell Friday"')
print()
print('If one weekday is significant:')
print('  -> Check if it holds across sub-periods (2019-2021 vs 2022-2025)')
print('  -> If it only worked in one period, it was luck, not a pattern')
print()
print('Sample size needed for confidence:')
print('  - 5+ years of data = ~1800 days = ~250 Mondays')
print('  - Below 100 observations per weekday = unreliable')

## 8. Robustness check - does the pattern hold across years?

**The real test:** if a weekday pattern is real, it should show up in most years. If it only appears in one or two years, it was luck.

In [ ]:
for sym in SYMBOLS:
    df = data[sym]
    df_year = df.copy()
    df_year['year'] = df_year.index.year

    pivot = df_year.groupby(['year', 'weekday'])['ret'].mean().unstack()
    pivot.columns = WEEKDAYS
    pivot = pivot * 100

    print(f'\n{sym} - mean daily return by year and weekday (%):')
    print(pivot.round(3).to_string())

    # count how many years each weekday was positive
    pos_years = (pivot > 0).sum()
    print(f'\n{sym} - how many years was each weekday positive?')
    print(pos_years.to_string())
    print(f'(out of {len(pivot)} years)')

## Final note

**What we learned:** day-of-week seasonality on BTC/ETH is well-studied and almost always **weak or non-existent** after 2020.

Why?

- Crypto is 24/7 - no forced weekend closure
- Institutional flow has grown, diluting retail patterns
- Any real pattern gets arbitraged away quickly

**Practical takeaway:** don't trade day-of-week. Use this notebook to confirm that intuition, then move to stronger signals (funding, OI, PCR) in the option_chain folder.